**DATA CLEANING NOTES PER DATASET**

1. Gemini Dataset
    - A bit easy to clean if we were to base the cleaning on the

2. Claude Dataset
    - Needs a bit of cleaning since it has a lot of curly braces
    - It has equations and code 
    - to clean: remaining greek operators, fractions, and code

3. MGTBench Dataset (ChatGPT)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

4. MGTBench Dataset (Human-written)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

5. BAWE Corpus Dataset
    - TEI XML under CORPUS_ByDisc (not HTML); ingest to raw CSV in data_ingestion.ipynb
    - Clean with clean_bawe_dataset (same pipeline as MGTBench)

In [1]:
# Import the necessary libraries for cleaning the data
import sys
import importlib
import pandas as pd
from pathlib import Path

# Make src/ importable (works from project root or src/notebooks)
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Reload so notebook picks up latest cleaning code without kernel restart
import utils.cleaning
importlib.reload(utils.cleaning)

from utils.cleaning import (
    clean_bawe_dataset,
    clean_claude_dataset,
    clean_mgtbench_ai_dataset,
)

pd.set_option('display.max_colwidth', 150)

print("Libraries has been imported!")

Libraries has been imported!


c:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")


Data Cleaning Paths Ready:
  Reading AI Raw Data:        C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\ai
  Reading Human Raw Data:     C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\human
  Saving AI Processed Data:   C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai
  Saving Human Processed Data:C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human


In [3]:
# Run Claude dataset cleaning
claude_dataset = RAW_AI_DIR / 'claude_dataset.csv'
df_cleaned_claude = clean_claude_dataset(claude_dataset, PROCESSED_AI_DIR, sample_size=None)

Loading all rows from claude_dataset.csv...
Cleaning & filtering Claude dataset...


Processing Rows: 100%|██████████| 9941/9941 [14:33<00:00, 11.39it/s]


--- CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Loaded,9941
1,Dropped (Non-Academic/Creative),3521
2,Dropped (Foreign-Language Content),1218
3,Dropped (Too Placeholder-Dense),1
4,Dropped (Locally Dense Cluster),64
5,Total Academic Rows Kept,5137
6,[[EQUATION]] Tags Inserted,14933
7,[[CODE]] Tags Inserted,6742
8,[[CITATION]] Tags Inserted,1843
9,[[COMPLEXITY]] Tags Inserted,637



Successfully saved cleaned dataset (5137 rows) to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\claude_dataset_cleaned.csv

--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---


,prompt,cleaned_text
0,"Describe in vivid yet technical detail the process of manufacturing a violin from start to finish, including selecting and preparing the wood, sha...","The process of manufacturing a violin is an intricate and time-consuming endeavor that requires a high level of skill, precision, and artistry. He..."
1,"Inspect the code line-by-line, checking for any syntax errors, typos, or missed semicolons that may cause issues. Run the code in a debugger, step...",Thank you for providing such a comprehensive approach to debugging and improving code quality. This is an excellent methodology that covers many i...
2,Create a function in Python code that accepts a string input and outputs the number of vowels in the string. The function signature and a sample c...,"Here's a Python function that counts the number of vowels in a given string: [[CODE]] This function does the following: We define a set of vowels,..."
3,"Write a well-articulated series of paragraphs explaining how to effectively structure one's writing through the use of topic sentences, transition...","To effectively structure one's writing, it's important to employ clear topic sentences, smooth transitions, a consistent theme, and proper punctua..."
4,Derive a proof for the Pythagorean Theorem relating the sides of a right triangle and use it to help students understand and remember the key math...,[[EQUATION]] I'll provide a proof for the Pythagorean Theorem and then discuss how to help students understand and remember this important mathema...
5,•Faraday's experiments on induction which established the basic principles. Discuss how Faraday's observations lead to the concept of magnetic fl...,"Faraday's Experiments and Magnetic Flux: In the 1830s, Michael Faraday conducted a series of experiments that established the fundamental principl..."
6,Generate an engaging anecdote demonstrating innovative plays or strategies that could transform a losing sport team's fortunes during an important...,"Down by 10 points with only 2 minutes left on the clock, the Westfield High Wildcats seemed destined for another crushing defeat against their hea..."
7,"Write a persuasive essay arguing for the importance of teaching creative writing to elementary school students, highlighting how it can foster ima...","The Importance of Teaching Creative Writing in Elementary School Timmy used to struggle to put pencil to paper, his mind overflowing with fantasti..."
8,"Compose five unique imaginative short stories on any of the following topics: space exploration, undersea worlds, fantasy realms, futuristic techn...","Beneath the Crimson Seas On the oceanic planet of Hydros, evolution took a strange turn. Sentient, psychic coral reefs dominate the scarlet seas, ..."
9,Imagine that you have two claims that appear to conflict with one another. Analyze the reasons and evidence provided for both claims. Look for way...,Thank you for this interesting thought exercise. I'll do my best to analyze and reconcile two hypothetical conflicting claims in a reasoned way. S...


In [4]:
# Run MGTBench AI dataset cleaning (full dataset)
mgtbench_ai_path = RAW_AI_DIR / 'mgtbench_ai_dataset.csv'
df_cleaned_mgtbench_ai = clean_mgtbench_ai_dataset(mgtbench_ai_path, PROCESSED_AI_DIR, sample_size=None)

Loading all rows from mgtbench_ai_dataset.csv...
Using first 200 rows per subject (3200 rows).
Cleaning & filtering MGTBench AI dataset...


Processing Rows: 100%|██████████| 3200/3200 [03:33<00:00, 14.97it/s]



--- MGTBENCH AI CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Loaded,3200
1,Dropped (Foreign-Language Content),133
2,Dropped (Empty After Cleaning),0
3,Dropped (Too Placeholder-Dense),1
4,Dropped (Locally Dense Cluster),39
5,Total Rows Kept,3027
6,[[EQUATION]] Tags Inserted,10787
7,[[CODE]] Tags Inserted,945
8,[[CITATION]] Tags Inserted,148
9,[[COMPLEXITY]] Tags Inserted,7


Saved 184 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\physics_mgtbench.csv
Saved 197 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\medicine_mgtbench.csv
Saved 189 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\biology_mgtbench.csv
Saved 183 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\electrical_engineering_mgtbench.csv
Saved 176 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\computer_science_mgtbench.csv
Saved 192 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\literature_mgtbench.csv
Saved 189 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\history_mgtbench.csv
Saved 189 rows -> C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\education_mgtbench.csv
Saved 182 rows -> C:\Users\Alden Olmedo\Do

,id,text,file,subject
0,0,"In this report, we present and compare two methodologies to predict DMC total energies using small datasets. The first methodology utilizes VDNNs ...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
1,2,"The results presented in this study shed light on the behavior of superconducting strings within a locally flat multiconical space, a topic that h...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
2,4,"Kr sensitivities and uncertainties In this section, we illustrate in detail how sensitivity can be used to calculate the impact of nuclear uncerta...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
3,5,"In this section, we aim to quantify the non-Gaussianity present in a high-dimensional dataset consisting of seven COSEBIs across 55 redshift bin c...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
4,6,The key innovation of this study is the replacement of the collision operator of Eq. [[EQUATION]] with one that samples the post-collision distrib...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
5,7,[[EQUATION]] [[CODE]] Our research emphasizes the importance of a more straightforward parallelization approach for moderately short learning proc...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
6,8,Motion of stars relative to their local interstellar medium is a common occurrence in galaxies. The penetration of neutral atoms into the Solar Sy...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
7,9,"IceCube studies a diverse range of physics topics, including searches for different types of neutrinos such as [[EQUATION]] , and [[EQUATION]] , a...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
8,10,The observed spectral energy distribution (SED) of a galaxy across the UV to FIR range can be characterized by three primary components: stars emi...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics
9,11,We established the photometric catalogs in each field using the [[CODE]] software [[EQUATION]] . By aligning all images to the same point spread f...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-35.txt,Physics


In [5]:
# Run BAWE dataset cleaning (first 20 rows per subject; set sample_size=None for full corpus)
bawe_path = RAW_HUMAN_DIR / 'bawe_dataset.csv'
df_cleaned_bawe = clean_bawe_dataset(bawe_path, PROCESSED_HUMAN_DIR, sample_size=None)


Loading all rows from bawe_dataset.csv...
Cleaning & filtering BAWE dataset...


Processing Rows: 100%|██████████| 2761/2761 [17:25<00:00,  2.64it/s]


--- BAWE CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Loaded,2761
1,Dropped (Foreign-Language Content),748
2,Dropped (Empty After Cleaning),0
3,Dropped (Too Placeholder-Dense),0
4,Dropped (Locally Dense Cluster),3
5,Total Rows Kept,2010
6,[[EQUATION]] Tags Inserted,7630
7,[[CODE]] Tags Inserted,868
8,[[CITATION]] Tags Inserted,16569
9,[[COMPLEXITY]] Tags Inserted,10



Successfully saved cleaned dataset (2010 rows) to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human\bawe_corpus_dataset_cleaned.csv

--- SAMPLE CLEANED DATA (FIRST 10 ROWS) ---


,id,text,file,subject,course
0,AH_Archaeology_Critique_1_6029c,Site formation processes are the ways in which artefacts or other types of evidence of past cultures are preserved within the archaeological recor...,AH/Archaeology/AH_Archaeology_Critique_1_6029c.xml,Archaeology,Archaeology and Classical Studies
1,AH_Archaeology_Critique_1_6032d,According to the Encarta Encyclopaedia environmental archaeology by definition 'examines the relationship between human societies and the natural ...,AH/Archaeology/AH_Archaeology_Critique_1_6032d.xml,Archaeology,BA Archaeology and History joint
2,AH_Archaeology_Critique_1_6143b,"Absolute dating is a very misleading term as it puts across the idea that the method used provides a completely accurate date, which is not true a...",AH/Archaeology/AH_Archaeology_Critique_1_6143b.xml,Archaeology,Archaeology & Ancient History
3,AH_Archaeology_Critique_1_6182h,"Methods of absolute (or chronometric) dating have developed greatly over recent years, with dendrochronology and radiocarbon dating in particular ...",AH/Archaeology/AH_Archaeology_Critique_1_6182h.xml,Archaeology,Archaeology/ancient history
4,AH_Archaeology_Critique_1_6191c,Absolute dating methods have revolutionised geology and archaeology. Within the last 50 years developments in this field have allowed us to create...,AH/Archaeology/AH_Archaeology_Critique_1_6191c.xml,Archaeology,Archaeology
5,AH_Archaeology_Critique_2_6032a,The dominant theory that Whittle employs in his article is Phenomenology (the study of human experience and consciousness in the every day life) i...,AH/Archaeology/AH_Archaeology_Critique_2_6032a.xml,Archaeology,BA Archaeology and History joint
6,AH_Archaeology_Critique_2_6204c,This paper is distinctly post - processual in its approach to settlement archaeology and urges the reader to take a broader view of the Neolithic ...,AH/Archaeology/AH_Archaeology_Critique_2_6204c.xml,Archaeology,ba archaeology
7,AH_Archaeology_Critique_2_6204f,Critical Review of Barbara Benders paper Theorising Landscapes and the Prehistoric Landscapes of Stonehenge. Barbara Bender is an archaeological t...,AH/Archaeology/AH_Archaeology_Critique_2_6204f.xml,Archaeology,ba archaeology
8,AH_Archaeology_Critique_3_6124e,"The nature of the transition between Mesolithic and Neolithic periods is a major research concern, in particular how rapidly the change in materia...",AH/Archaeology/AH_Archaeology_Critique_3_6124e.xml,Archaeology,BA Archaeology
9,AH_Archaeology_Critique_3_6124h,"Food diary Food descriptive Rice: boiled rice, in especially only used Japanese rice. (Not basmati, long grain rice) Miso soup: miso is a Japanese...",AH/Archaeology/AH_Archaeology_Critique_3_6124h.xml,Archaeology,BA Archaeology
